In [5]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from statsmodels.tsa.arima.model import ARIMA
from prophet import Prophet
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense

In [7]:
target_variables = ["uzman", "pratisyen", "asistan", "ebe", "diger", "hemsire", "yatak", "hastane"]

feature_map = {
    "uzman": ["yıl", "nufus", "hastane", "yatak", "asistan", "pratisyen", "hemsire", "ebe", "diger"],
    "pratisyen": ["yıl", "nufus", "hastane", "yatak", "asistan", "uzman", "hemsire", "ebe", "diger"],
    "asistan": ["yıl", "nufus", "hastane", "yatak", "uzman", "pratisyen", "hemsire", "ebe", "diger"],
    "ebe": ["yıl", "nufus", "hastane", "yatak", "asistan", "uzman", "hemsire", "pratisyen", "diger"],
    "diger": ["yıl", "nufus", "hastane", "yatak", "asistan", "pratisyen", "hemsire", "ebe", "uzman"],
    "hemsire": ["yıl", "nufus", "hastane", "yatak", "asistan", "uzman", "pratisyen", "ebe", "diger"],
    "yatak": ["yıl", "nufus", "hastane", "hemsire", "asistan", "uzman", "pratisyen", "ebe", "diger"],
    "hastane": ["yıl", "nufus", "yatak", "hemsire", "asistan", "uzman", "pratisyen", "ebe", "diger"]
}

df = pd.read_csv("all_data_ratio.csv")

results = []

In [9]:
# --- Modellerin Fonksiyonları ---

def forecast_arima(series):
    model = ARIMA(series, order=(1, 1, 1)).fit()
    forecast = model.forecast()[0]
    return forecast

def forecast_exp_smoothing(series):
    model = ExponentialSmoothing(series, trend='add', seasonal=None).fit()
    forecast = model.forecast(1)[0]
    return forecast

def forecast_prophet(series):
    prophet_df = pd.DataFrame({'ds': pd.date_range(start='2002', periods=len(series), freq='Y'),
                               'y': series.values})
    model = Prophet(yearly_seasonality=False)
    model.fit(prophet_df)
    future = model.make_future_dataframe(periods=1, freq='Y')
    forecast = model.predict(future).iloc[-1]['yhat']
    return forecast

def forecast_lstm(X, y):
    scaler = MinMaxScaler()
    X_scaled = scaler.fit_transform(X)
    y_scaled = scaler.fit_transform(y.reshape(-1, 1))

    X_lstm = np.reshape(X_scaled, (X_scaled.shape[0], 1, X_scaled.shape[1]))

    model = Sequential([
        LSTM(50, input_shape=(X_lstm.shape[1], X_lstm.shape[2])),
        Dense(1)
    ])
    model.compile(optimizer='adam', loss='mse')
    model.fit(X_lstm[:-1], y_scaled[:-1], epochs=100, verbose=0)

    X_pred = np.reshape(X_scaled[-1:], (1, 1, X_scaled.shape[1]))
    prediction_scaled = model.predict(X_pred)[0][0]
    prediction = scaler.inverse_transform([[prediction_scaled]])[0][0]
    return prediction

In [14]:
# --- Tahmin Döngüsü ---

for il in df['il_adı'].unique():
    il_data = df[df['il_adı'] == il]
    for target in target_variables:
        features = feature_map[target]
        columns_to_use = list(set(features + [target]))  # Hedef değişkeni de dahil et
        il_filtered = il_data[columns_to_use]
        il_filtered = il_filtered[il_filtered['yıl'] <= 2022]

        if len(il_filtered) < 5:
            continue

        y_series = il_filtered[target]

        try:
            arima_pred = forecast_arima(y_series)
        except:
            arima_pred = None

        try:
            prophet_pred = forecast_prophet(y_series)
        except:
            prophet_pred = None

        try:
            exp_pred = forecast_exp_smoothing(y_series)
        except:
            exp_pred = None

        try:
            X = il_filtered.drop(columns=[target]).values
            y = y_series.values
            lstm_pred = forecast_lstm(X, y)
        except:
            lstm_pred = None

        results.append({
            'il': il,
            'target': target,
            'arima': arima_pred,
            'prophet': prophet_pred,
            'exp_smoothing': exp_pred,
            'lstm': lstm_pred
        })

C:\Users\Bedirhan\anaconda3\Lib\site-packages\statsmodels\tsa\statespace\sarimax.py:966: UserWarning: Non-stationary starting autoregressive parameters found. Using zeros as starting parameters.
  warn('Non-stationary starting autoregressive parameters'
C:\Users\Bedirhan\anaconda3\Lib\site-packages\statsmodels\tsa\statespace\sarimax.py:978: UserWarning: Non-invertible starting MA parameters found. Using zeros as starting parameters.
  warn('Non-invertible starting MA parameters found.'
C:\Users\Bedirhan\AppData\Local\Temp\ipykernel_528\775311501.py:14: FutureWarning: 'Y' is deprecated and will be removed in a future version, please use 'YE' instead.
  prophet_df = pd.DataFrame({'ds': pd.date_range(start='2002', periods=len(series), freq='Y'),
01:15:14 - cmdstanpy - INFO - Chain [1] start processing
01:15:14 - cmdstanpy - INFO - Chain [1] done processing
C:\Users\Bedirhan\anaconda3\Lib\site-packages\prophet\forecaster.py:1854: FutureWarning: 'Y' is deprecated and will be removed in a fu

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 325ms/step


C:\Users\Bedirhan\AppData\Local\Temp\ipykernel_528\775311501.py:14: FutureWarning: 'Y' is deprecated and will be removed in a future version, please use 'YE' instead.
  prophet_df = pd.DataFrame({'ds': pd.date_range(start='2002', periods=len(series), freq='Y'),
01:15:29 - cmdstanpy - INFO - Chain [1] start processing
01:15:29 - cmdstanpy - INFO - Chain [1] done processing
C:\Users\Bedirhan\anaconda3\Lib\site-packages\prophet\forecaster.py:1854: FutureWarning: 'Y' is deprecated and will be removed in a future version, please use 'YE' instead.
  dates = pd.date_range(
C:\Users\Bedirhan\anaconda3\Lib\site-packages\keras\src\layers\rnn\rnn.py:200: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 323ms/step


C:\Users\Bedirhan\AppData\Local\Temp\ipykernel_528\775311501.py:14: FutureWarning: 'Y' is deprecated and will be removed in a future version, please use 'YE' instead.
  prophet_df = pd.DataFrame({'ds': pd.date_range(start='2002', periods=len(series), freq='Y'),
01:15:45 - cmdstanpy - INFO - Chain [1] start processing
01:15:45 - cmdstanpy - INFO - Chain [1] done processing
C:\Users\Bedirhan\anaconda3\Lib\site-packages\prophet\forecaster.py:1854: FutureWarning: 'Y' is deprecated and will be removed in a future version, please use 'YE' instead.
  dates = pd.date_range(
C:\Users\Bedirhan\anaconda3\Lib\site-packages\keras\src\layers\rnn\rnn.py:200: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


...

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 378ms/step


C:\Users\Bedirhan\anaconda3\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: An unsupported index was provided and will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)
C:\Users\Bedirhan\anaconda3\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: An unsupported index was provided and will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)
C:\Users\Bedirhan\anaconda3\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: An unsupported index was provided and will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)
C:\Users\Bedirhan\anaconda3\Lib\site-packages\statsmodels\tsa\statespace\sarimax.py:966: UserWarning: Non-stationary starting autoregressive parameters found. Using zeros as starting parameters.
  warn('Non-stationary starting autoregressive parameters'
C:\Users\Bedirhan\anaconda3\Lib\site-packages\statsmodels\tsa\statespace\sarimax.py:978: UserWarning: Non-invertible start

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 286ms/step


C:\Users\Bedirhan\anaconda3\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: An unsupported index was provided and will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)
C:\Users\Bedirhan\anaconda3\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: An unsupported index was provided and will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)
C:\Users\Bedirhan\anaconda3\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: An unsupported index was provided and will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)
C:\Users\Bedirhan\anaconda3\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:836: ValueWarning: No supported index is available. Prediction results will be given with an integer index beginning at `start`.
  return get_prediction_index(
C:\Users\Bedirhan\anaconda3\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:836: FutureWarning: No supported index is available. In 

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 356ms/step


In [ ]:
# Sonuçları DataFrame olarak kaydet
results_df = pd.DataFrame(results)
results_df.to_csv("tahmin_sonuclari_2023.csv", index=False)

In [ ]:
# Daha önce oluşturulan sonuçları içeren df
results_df = pd.read_csv("tahmin_sonuclari_2023.csv")

# Pivot yaparak şehir bazlı görünüm oluştur
pivot_df = results_df.pivot(index='il', columns='target', values=['arima', 'prophet', 'exp_smoothing', 'lstm'])

# Çok katmanlı sütun isimlerini tek katmana indir
pivot_df.columns = [f"{model}_{var}" for model, var in pivot_df.columns]

# Şehir ismini başa al
pivot_df.reset_index(inplace=True)

# Sonucu CSV olarak kaydet
pivot_df.to_csv("sehir_bazli_tahminler_2023.csv", index=False)

In [ ]:
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error, mean_absolute_percentage_error

# Tahminleri oku
pred_df = pd.read_csv("tahmin_sonuclari_2023.csv")

# Gerçek veriyi oku
real_df = pd.read_csv("all_data_ratio.csv")
real_df = real_df[real_df['yıl'] == 2023]

# Sonuçları saklamak için liste
metrics_list = []

# Her hedef değişken ve model için skor hesapla
models = ["arima", "prophet", "exp_smoothing", "lstm"]
target_vars = pred_df['target'].unique()

for target in target_vars:
    for model in models:
        # Model tahminlerini ve gerçek veriyi eşleştir
        pred_subset = pred_df[pred_df['target'] == target][['il', model]].dropna()
        real_subset = real_df[['il_adı', target]].rename(columns={'il_adı': 'il'})
        merged = pd.merge(pred_subset, real_subset, on='il')
        
        if len(merged) < 5:  # Sağlıklı istatistik için minimum örnek
            continue
        
        y_true = merged[target].values
        y_pred = merged[model].values

        # Metrikleri hesapla
        r2 = r2_score(y_true, y_pred)
        mae = mean_absolute_error(y_true, y_pred)
        rmse = mean_squared_error(y_true, y_pred, squared=False)
        mape = mean_absolute_percentage_error(y_true, y_pred)

        metrics_list.append({
            'target': target,
            'model': model,
            'R2': round(r2, 4),
            'MAE': round(mae, 2),
            'RMSE': round(rmse, 2),
            'MAPE (%)': round(mape * 100, 2)
        })

# Sonuçları DataFrame'e çevir ve kaydet
metrics_df = pd.DataFrame(metrics_list)
metrics_df.to_csv("model_dogruluk_skorlari.csv", index=False)

# Ekrana yazdır
import ace_tools as tools; tools.display_dataframe_to_user(name="Model Doğruluk Skorları", dataframe=metrics_df)

In [25]:
import pandas as pd
from sklearn.metrics import mean_absolute_percentage_error

# Bölge haritası
bolge_haritasi = {
    'Marmara': ["İstanbul", "Edirne", "Kırklareli", "Tekirdağ", "Kocaeli", "Sakarya", "Yalova", "Bilecik", "Balıkesir", "Çanakkale", "Bursa"],
    'Ege': ["İzmir", "Aydın", "Muğla", "Manisa", "Afyonkarahisar", "Uşak", "Kütahya", "Denizli"],
    'Akdeniz': ["Antalya", "Mersin", "Adana", "Hatay", "Kahramanmaraş", "Osmaniye", "Isparta", "Burdur"],
    'İç Anadolu': ["Ankara", "Konya", "Kayseri", "Eskişehir", "Nevşehir", "Aksaray", "Kırıkkale", "Niğde", "Karaman", "Sivas", "Yozgat", "Kırşehir", "Çankırı"],
    'Karadeniz': ["Samsun", "Trabzon", "Ordu", "Giresun", "Rize", "Artvin", "Tokat", "Amasya", "Bartın", "Zonguldak", "Karabük", "Sinop", "Çorum", "Bayburt", "Gümüşhane", "Kastamonu", "Bolu", "Düzce"],
    'Doğu Anadolu': ["Erzurum", "Van", "Malatya", "Elazığ", "Kars", "Ağrı", "Bitlis", "Iğdır", "Tunceli", "Ardahan", "Erzincan", "Bingöl", "Muş", "Hakkari"],
    'Güneydoğu Anadolu': ["Diyarbakır", "Gaziantep", "Şanlıurfa", "Mardin", "Batman", "Siirt", "Şırnak", "Adıyaman", "Kilis"]
}

# Bölge eşlemesi
bolge_df = pd.DataFrame([
    {'il': sehir, 'bolge': bolge}
    for bolge, iller in bolge_haritasi.items()
    for sehir in iller
])

# Verileri oku
tahminler = pd.read_csv("tahmin_sonuclari_2023.csv")
gercekler = pd.read_csv("all_data_ratio.csv")

# Gerçek verileri 2023 yılı için al ve uzun forma çevir
gercekler = gercekler[gercekler["yıl"] == 2023]
gercekler.rename(columns={"il_adı": "il"}, inplace=True)
gercekler = gercekler.melt(id_vars=["il"], var_name="target", value_name="gercek")

# Birleştir
birlesik = pd.merge(tahminler, gercekler, on=["il", "target"], how="inner")
birlesik = pd.merge(birlesik, bolge_df, on="il", how="left")

# Bölge bazlı prophet/lstm ve gerçek toplamları
bolgesel_ortalamalar = birlesik.groupby(["bolge", "target"]).agg({
    "gercek": "sum",
    "prophet": "sum",
    "lstm": "sum"
}).reset_index()

# Doğruluk hesapla
bolgesel_ortalamalar["prophet_accuracy_%"] = 100 - mean_absolute_percentage_error(
    bolgesel_ortalamalar["gercek"], bolgesel_ortalamalar["prophet"]) * 100
bolgesel_ortalamalar["lstm_accuracy_%"] = 100 - mean_absolute_percentage_error(
    bolgesel_ortalamalar["gercek"], bolgesel_ortalamalar["lstm"]) * 100

# Toplam doğruluk
toplam_prophet_accuracy = 100 - mean_absolute_percentage_error(
    birlesik["gercek"], birlesik["prophet"]) * 100
toplam_lstm_accuracy = 100 - mean_absolute_percentage_error(
    birlesik["gercek"], birlesik["lstm"]) * 100

# CSV olarak kaydet
csv_path = "bolgesel_karsilastirma.csv"
bolgesel_ortalamalar.to_csv(csv_path, index=False)

# Göster
import ace_tools as tools
tools.display_dataframe_to_user(name="Bölgesel Prophet ve LSTM Karşılaştırması", dataframe=bolgesel_ortalamalar)

print("Toplam Prophet doğruluğu: %", round(toplam_prophet_accuracy, 2))
print("Toplam LSTM doğruluğu: %", round(toplam_lstm_accuracy, 2))


ModuleNotFoundError: No module named 'ace_tools'